# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ubaidrees/flyrank-ml-tasks/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

Method: Random Forest Classifier. The lane guide's own verified benchmark on this dataset shows Random Forest reaching Precision@50 = 0.740 vs. the baseline rule's 0.240 and Logistic Regression's 0.400 — a real, evidenced jump, not a guess. It also fits what ML-07's signal audit found: staleness and CTR effects were non-linear (staleness reversed at 180+ days; CTR only separated cleanly after removing low-volume outliers), which tree ensembles capture naturally and linear models miss.

In [7]:
import pandas as pd
import numpy as np

df = pd.read_csv("https://raw.githubusercontent.com/Ubaidrees/flyrank-ml-tasks/main/data/raw/content_refresh_anonymized.csv")

df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
df = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].copy()
df = df.drop_duplicates(subset='content_id')

feature_cols = [c for c in [
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d',
    'content_age_days', 'days_since_last_update',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate',
    'word_count', 'ai_traffic_pct'
] if c in df.columns]

print("Using features:", feature_cols)
print(f"Rows after filtering: {len(df)}")
print(f"Positive rate (declining): {df['is_declining_label'].mean():.3f}")

Using features: ['impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'word_count', 'ai_traffic_pct']
Rows after filtering: 30000
Positive rate (declining): 0.542


## 2. Split design

Split: client-holdout (GroupShuffleSplit), 80/20. Fixes a weakness ML-07 surfaced directly — client_7f2253d7e2 accounted for 6 of the top 10 baseline picks. A row-random split risks leaking that client's pages into both train and test, letting the model memorize client quirks instead of learning generalizable signal. This split guarantees zero client overlap between train and test.

In [8]:
from sklearn.model_selection import GroupShuffleSplit

X = df[feature_cols].fillna(0)
y = df['is_declining_label']
groups = df['client_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
df_test = df.iloc[test_idx].copy()

overlap = set(df.iloc[train_idx]['client_id']) & set(df.iloc[test_idx]['client_id'])
print(f"Train rows: {len(X_train)}, Test rows: {len(X_test)}")
print(f"Client overlap between train/test: {len(overlap)} (must be 0)")

Train rows: 23837, Test rows: 6163
Client overlap between train/test: 0 (must be 0)


## 3. Train + compare vs my baseline

Comparison design: same review budget, not a fixed Precision@50. The ML-07 baseline flags very few pages (10 of 30,000 overall), so a literal Precision@50 is unreachable for it. Instead, the model is evaluated on its own top-k picks, where k equals however many rows the baseline flagged in this same test set — an apples-to-apples "same capacity to review" comparison.

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

baseline_flag = (
    (df_test['days_since_last_update'] >= 180) &
    (df_test['impressions_90d'] >= 500) &
    (df_test['avg_position'] > 0) & (df_test['avg_position'] <= 20) &
    (df_test['ctr'] < 0.5)
)
k = int(baseline_flag.sum())
baseline_precision = y_test[baseline_flag.values].mean() if k > 0 else np.nan
print(f"Baseline flagged {k} of {len(df_test)} test rows")
print(f"Baseline precision: {baseline_precision:.3f}" if k > 0 else "Baseline flagged 0 rows in test — cannot score directly.")

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    random_state=42,
    class_weight='balanced'
)
model.fit(X_train, y_train)
model_proba = model.predict_proba(X_test)[:, 1]
model_auc = roc_auc_score(y_test, model_proba)

k_eval = max(k, 1)
top_k_idx = np.argsort(model_proba)[::-1][:k_eval]
model_precision_at_k = y_test.iloc[top_k_idx].mean()

print(f"\nModel ROC AUC: {model_auc:.3f}")
print(f"Model Precision@{k_eval} (same budget as baseline): {model_precision_at_k:.3f}")

comparison = pd.DataFrame({
    'Method': ['Baseline rule (ML-07)', f'Random Forest (k={k_eval})'],
    f'Precision@{k_eval}': [baseline_precision, model_precision_at_k],
})
print("\n=== MODEL vs BASELINE ===")
print(comparison.to_string(index=False))

Baseline flagged 0 of 6163 test rows
Baseline flagged 0 rows in test — cannot score directly.

Model ROC AUC: 0.604
Model Precision@1 (same budget as baseline): 1.000

=== MODEL vs BASELINE ===
               Method  Precision@1
Baseline rule (ML-07)          NaN
  Random Forest (k=1)          1.0


In [10]:
# Baseline evaluated on the FULL dataset — legitimate since a hand-written rule
# has no parameters fit to data, unlike the model (which only saw training rows)
baseline_flag_full = (
    (df['days_since_last_update'] >= 180) &
    (df['impressions_90d'] >= 500) &
    (df['avg_position'] > 0) & (df['avg_position'] <= 20) &
    (df['ctr'] < 0.5)
)
k_full = int(baseline_flag_full.sum())
baseline_precision_full = df.loc[baseline_flag_full, 'is_declining_label'].mean()
print(f"Baseline flagged {k_full} of {len(df)} total rows (full dataset)")
print(f"Baseline precision (full dataset): {baseline_precision_full:.3f}")

# Model: same total budget (k_full), but scored on the TEST set only — stricter on the model
top_k_full_idx = np.argsort(model_proba)[::-1][:k_full]
model_precision_at_k_full = y_test.iloc[top_k_full_idx].mean()
print(f"\nModel Precision@{k_full} (test set only, same total budget): {model_precision_at_k_full:.3f}")

fair_comparison = pd.DataFrame({
    'Method': ['Baseline rule (full dataset)', f'Random Forest (test only, k={k_full})'],
    'Precision': [baseline_precision_full, model_precision_at_k_full],
})
print("\n=== FAIR COMPARISON ===")
print(fair_comparison.to_string(index=False))

Baseline flagged 10 of 30000 total rows (full dataset)
Baseline precision (full dataset): 1.000

Model Precision@10 (test set only, same total budget): 0.400

=== FAIR COMPARISON ===
                         Method  Precision
   Baseline rule (full dataset)        1.0
Random Forest (test only, k=10)        0.4


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## 4. Errors and interpretation

### Model vs. baseline

The Random Forest achieved a **ROC AUC of 0.604**, indicating that it learned useful signal beyond random guessing, but it did **not outperform** the handcrafted baseline rule under the precision comparison.

The baseline rule flagged only **10 pages** across the entire dataset and achieved **100% precision** because those rules are extremely strict. In comparison, the Random Forest achieved **Precision@10 = 0.400** on the held-out test set.

This suggests that, for this dataset, the manually designed business rules remain more reliable when the review budget is very small. The machine learning model is more flexible but also introduces more false positives.

### Feature interpretation

Permutation importance showed that the model relied primarily on:

- impressions_90d
- content_age_days
- avg_position
- clicks_90d
- ctr

Interestingly, **days_since_last_update** showed a negative permutation importance. This indicates that, after the other features were included, this feature did not improve predictive performance and may contain redundant information already captured by content age or engagement-related variables.

### Error analysis

The model produced:

- False Negatives: **1281**
- False Positives: **1308**

The errors are relatively balanced, suggesting the model is not strongly biased toward predicting only one class. However, the relatively large number of both error types explains the modest ROC AUC and lower Precision@10 compared with the baseline.

### Conclusion

For this dataset, the handcrafted baseline remains the better choice when only a very small number of pages can be reviewed. The Random Forest successfully learned useful patterns but did not surpass the precision of the expert-designed rules.

Future improvements could include additional feature engineering, hyperparameter tuning, probability calibration, or testing Gradient Boosting models while maintaining the same grouped validation strategy.

In [11]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(model, X_test, y_test, n_repeats=20, random_state=42, scoring='roc_auc')
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std
}).sort_values('importance_mean', ascending=False)
print("Permutation importance:")
print(importance_df.to_string(index=False))

y_pred = (model_proba >= 0.5).astype(int)
df_test_result = df_test.copy()
df_test_result['y_true'] = y_test.values
df_test_result['y_pred'] = y_pred

fn = df_test_result[(df_test_result['y_true']==1) & (df_test_result['y_pred']==0)]
fp = df_test_result[(df_test_result['y_true']==0) & (df_test_result['y_pred']==1)]
print(f"\nFalse negatives: {len(fn)}  |  False positives: {len(fp)}")

Permutation importance:
               feature  importance_mean  importance_std
       impressions_90d         0.067481        0.003678
      content_age_days         0.021599        0.006420
          avg_position         0.016484        0.001410
            clicks_90d         0.013712        0.001148
                   ctr         0.007939        0.001316
           scroll_rate         0.006834        0.001140
          sessions_90d         0.005586        0.000703
       engagement_rate         0.003376        0.000371
       ai_sessions_90d         0.000068        0.000083
        ai_traffic_pct         0.000021        0.000058
            word_count        -0.004637        0.001866
days_since_last_update        -0.017593        0.002071

False negatives: 1281  |  False positives: 1308


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.